In [1]:
import pm4py
import pandas as pd

from IPython.display import Image, display
from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness_evaluator
from pm4py.algo.evaluation.precision import algorithm as precision_evaluator
from pm4py.algo.evaluation.generalization import algorithm as generalization_evaluator
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_evaluator

In [ ]:
# Load the XES file
log = pm4py.read_xes("BPI Challenge 2017.xes")

# Convert the log to a DataFrame
log_df = pm4py.convert_to_dataframe(log)

## Preprocessing

In [ ]:
# Keep only events with lifecycle=complete
if "lifecycle:transition" in log_df.columns:
    before_events = len(log_df)
    before_cases = log_df["case:concept:name"].nunique()

    log_filt_complete_events = log_df[
        log_df["lifecycle:transition"].str.lower() == "complete"
    ].copy()

    log_filt = log_filt_complete_events.copy()

    after_events = len(log_filt_complete_events)
    after_cases = log_filt_complete_events["case:concept:name"].nunique()

    print("Events before:", before_events, "| after:", after_events)
    print("Cases before:", before_cases, "| after:", after_cases)
    print("Lifecycle values:", log_df["lifecycle:transition"].dropna().value_counts().head(10))
else:
    print("No lifecycle:transition column found in this log.")

In [ ]:
# Preprocessing: keep cases that contain a final application outcome

case_col = "case:concept:name"
activity_col = "concept:name"
timestamp_col = "time:timestamp"

# ctivities identifying completed applications
completed_end_activities = {
    "A_Cancelled",
    "A_Denied",
    "A_Pending",
}


# Keep cases that contain at least one outcome event
outcome_cases = log_filt[log_filt[activity_col].isin(completed_end_activities)][case_col].unique()
log_filt = log_filt[log_filt[case_col].isin(outcome_cases)].copy()

print("Cases in source log:", log_filt[case_col].nunique())
print("Cases with outcome event:", len(outcome_cases))
print("Cases after outcome-case filter:", log_filt[case_col].nunique())

## BPMN

In [ ]:
## Build BPMN

# Discover BPMN
bpmn_unf = pm4py.discover_bpmn_inductive(log_filt, noise_threshold=0.3)
print("BPMN discovered from unfiltered log.")

# Safe BPMN
bpmn_unf_path = "results/BPMN/bpmn_unfiltered.bpmn"
pm4py.write_bpmn(bpmn_unf, bpmn_unf_path)
print("Saved:", bpmn_unf_path)

# Safe BPMN image
bpmn_unf_png = "results/BPMN/bpmn_unfiltered.png"
pm4py.save_vis_bpmn(bpmn_unf, bpmn_unf_png)

print("BPMN (unfiltered):")
display(Image(filename=bpmn_unf_png))

In [ ]:
# BPMN quality metrics

# Use PM4Py evaluators. Therefore, convert BPMN to petrinet first
net_bpmn, im_bpmn, fm_bpmn = pm4py.convert_to_petri_net(bpmn_unf)

# Fitness on lifecycle-complete baseline
fit_bpmn = replay_fitness_evaluator.apply(
    log_filt_complete_events,
    net_bpmn,
    im_bpmn,
    fm_bpmn,
    variant=replay_fitness_evaluator.Variants.TOKEN_BASED,
)

# Log fitness (replay)
log_fitness_bpmn = fit_bpmn.get("log_fitness", fit_bpmn.get("average_trace_fitness"))

# Percentage of fitting traces
replay_ratio_bpmn = fit_bpmn.get("perc_fit_traces", fit_bpmn.get("percentage_of_fitting_traces"))
if replay_ratio_bpmn is not None and replay_ratio_bpmn > 1:
    replay_ratio_bpmn = replay_ratio_bpmn / 100.0

# Precision
precision_bpmn = precision_evaluator.apply(
    log_filt_complete_events,
    net_bpmn,
    im_bpmn,
    fm_bpmn,
    variant=precision_evaluator.Variants.ETCONFORMANCE_TOKEN,
)

print("BPMN (converted) - Replay ratio:", round(float(replay_ratio_bpmn), 4))
print("BPMN (converted) - Log fitness:", round(float(log_fitness_bpmn), 4))
print("BPMN (converted) - Precision:", round(float(precision_bpmn), 4))

# Generalization and simplicity
generalization_bpmn = generalization_evaluator.apply(log_filt_complete_events, net_bpmn, im_bpmn, fm_bpmn)
simplicity_bpmn = simplicity_evaluator.apply(net_bpmn)

print("BPMN (converted) - Generalization:", round(float(generalization_bpmn), 4))
print("BPMN (converted) - Simplicity:", round(float(simplicity_bpmn), 4))